# Regression Trees, Random Forest and Gradient Boosting – Concepts

This notebook summarises the concepts and adds explanations and small examples. All examples use small hand-made numbers or simulated data – no data set is loaded.

**Contents**
1. Regression trees: partitioning the feature space
2. How a split is chosen (worked example)
3. Pruning
4. Hyperparameters in scikit-learn
5. Ensembles: why averaging many models helps
6. Bagging and out-of-bag observations
7. Random forest
8. Gradient boosting
9. Regression models compared

## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold

# Ignore warnings
warnings.filterwarnings('ignore')

## 1. Regression trees: partitioning the feature space

A regression tree asks a sequence of yes/no questions (e.g. $x_1 \le 3.99$?). Geometrically, it cuts the feature space
into **boxes** and predicts **one constant per box** – the mean of the training points in that box.

The simulation below creates data with exactly such a box structure and fits a tree with three splits.

In [ ]:
# Simulate 300 points with two features x1 and x2
rng = np.random.default_rng(0)
X = rng.uniform(0, 10, (300, 2))
x1 = X[:, 0]
x2 = X[:, 1]

# y is constant in four boxes of the feature space (plus noise)
y = np.full(300, 55.0)
y[(x1 > 4) & (x2 > 6) & (x1 <= 7.5)] = 40
y[(x1 > 4) & (x2 <= 6)] = 25
y[x1 <= 4] = 10
y = y + rng.normal(0, 3, 300)

# Regression tree with four leaves (three splits)
reg_tree = DecisionTreeRegressor(max_leaf_nodes=4, random_state=0)
reg_tree.fit(X, y)
print(tree.export_text(reg_tree, feature_names=['x1', 'x2'], decimals=2))

# Figure showing the points, coloured by the prediction of the tree
plt.scatter(x1, x2, c=reg_tree.predict(X))
plt.colorbar(label='prediction y_hat')
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('Regression tree: one constant per box')
plt.show()

## 2. How a split is chosen

1. Pick a feature (e.g. area) and a threshold (e.g. 80 m²): smaller values go left, the rest right.
2. Compute the mean of $y$ in each child and the squared errors around it (SSE).
3. Try **all** features and thresholds; keep the split with the smallest total SSE.

$$\min_{j,\,s}\; \Big[\sum_{x_i \in R_1(j,s)}(y_i-\bar y_{R_1})^2 + \sum_{x_i \in R_2(j,s)}(y_i-\bar y_{R_2})^2\Big]$$

Leaf prediction = average $y$ of the training points in the leaf. **Greedy and recursive:** split, then repeat in each
child until a stopping rule applies.

**Worked example:** $x = 1, 2, 3, 4, 5$ → $y = 10, 12, 14, 30, 34$. Root node: $\bar y = 20$,
SSE $= 100 + 64 + 36 + 100 + 196 = 496$.

In [ ]:
# Worked example: five observations
x = np.array([1, 2, 3, 4, 5])
y = np.array([10, 12, 14, 30, 34])


# Function: computes the sum of squared deviations of v from its mean
def sse(v):
    """Sum of squared errors around the mean."""
    return np.sum((v - v.mean()) ** 2)


# Root node
print('Root: mean =', y.mean(), ' SSE =', sse(y))

# SSE of all possible splits (thresholds between neighbouring x values)
for s in [1.5, 2.5, 3.5, 4.5]:
    left = y[x <= s]
    right = y[x > s]
    print(
        'x <=',
        s,
        ': left',
        left,
        'mean',
        left.mean(),
        '| right',
        right,
        'mean',
        right.mean(),
        '| SSE =',
        sse(left) + sse(right),
    )

**Decision:** the best split is $x \le 3.5$ (between the 3rd and 4th observation). The SSE drops from 496 to 16
(−97 %). Leaf predictions: 12 and 32. The procedure is then repeated in each child node.

## 3. Pruning

A tree grown until every leaf is (almost) pure has a very high variance. **Cost-complexity pruning** (Breiman, Friedman,
Olshen & Stone, 1984) balances fit and size:

$$R_\alpha(T) = \underbrace{\sum_{m=1}^{|T|}\sum_{x_i \in R_m}(y_i - \bar y_{R_m})^2}_{\text{error on the training data}}
+ \alpha\,|T| \qquad (|T| = \text{number of leaves})$$

In words: training error plus a fee $\alpha$ for every leaf. Higher fee → smaller tree.
- $\alpha = 0$: no penalty, largest subtree (may overfit),
- $\alpha$ large: very small tree (may underfit),
- choose $\alpha$ by cross-validation (`ccp_alpha` in scikit-learn).

In [ ]:
# Simulate 300 points of a wavy curve with noise
rng = np.random.default_rng(1)
x1 = rng.uniform(0, 10, 300)
y1 = np.sin(x1) * 3 + 0.3 * x1 + rng.normal(0, 1, 300)
X1 = x1.reshape(-1, 1)

# Candidate values of alpha (from the full tree)
full_tree = DecisionTreeRegressor(random_state=0)
alphas = full_tree.cost_complexity_pruning_path(X1, y1).ccp_alphas

# For each alpha: number of leaves, training error and 5-fold CV error
cv = KFold(n_splits=5, shuffle=True, random_state=0)
leaves = []
train_mse = []
cv_mse = []
for alpha in alphas:
    model = DecisionTreeRegressor(ccp_alpha=alpha, random_state=0)
    scores = cross_val_score(
        model, X1, y1, cv=cv, scoring='neg_mean_squared_error'
    )
    cv_mse.append(-scores.mean())
    model.fit(X1, y1)
    leaves.append(model.get_n_leaves())
    train_mse.append(np.mean((y1 - model.predict(X1)) ** 2))

# Tree size with the smallest CV error
best = leaves[np.argmin(cv_mse)]
print('Optimal size:', best, 'leaves')

# Figure showing the training and CV error by number of leaves
plt.plot(leaves, train_mse, marker='.', label='training error')
plt.plot(leaves, cv_mse, marker='.', label='5-fold CV error')
plt.axvline(best, color='black', linestyle=':', label='optimal size')
plt.xscale('log')
plt.xlabel('number of leaves (tree complexity, log scale)')
plt.ylabel('MSE')
plt.legend()
plt.show()

## 4. Hyperparameters in scikit-learn

| parameter | meaning | increasing it … |
|---|---|---|
| `max_depth` | maximum depth of the tree | more variance, less bias |
| `min_samples_split` | min. observations to split a node | less variance, more bias |
| `min_samples_leaf` | min. observations in a leaf | less variance, more bias |
| `max_leaf_nodes` | maximum number of leaves | more variance, less bias |
| `max_features` | features considered per split | less randomisation: stronger but more correlated trees (RF) |
| `ccp_alpha` | cost-complexity penalty $\alpha$ | smaller tree, less variance |
| `criterion` | `squared_error` (regression), `gini` / `entropy` (classification) | — |

## 5. Ensembles: why averaging many models helps

Assume $B$ models with equal variance $\sigma^2$ and common pairwise correlation $\rho$. The variance of their average is

$$\operatorname{Var}\Big(\frac{1}{B}\sum_{b=1}^{B}\hat f_b\Big) = \underbrace{\rho\,\sigma^2}_{\text{shared part}}
+ \underbrace{\frac{1-\rho}{B}\,\sigma^2}_{\text{independent part}}$$

Averaging shrinks only the independent part; the shared part $\rho\sigma^2$ remains. Example: $\sigma^2 = 1$,
$\rho = 0.2$, $B = 100$ → $0.2 + 0.8/100 = 0.208$: averaging removes about 80 % of the variance, but never the
$\rho\sigma^2$ part.

**Key idea:** bagging mainly reduces **variance**; boosting reduces systematic error step by step and can affect both
**bias and variance**.

In [ ]:
# Variance of the average of B models (sigma² = 1) for different rho
B = np.arange(1, 201)

# Figure showing the variance of the average by B for different rho
for rho in [0.0, 0.2, 0.5, 0.8]:
    plt.plot(B, rho + (1 - rho) / B, label='rho = ' + str(rho))
plt.xscale('log')
plt.xlabel('number of models B (log scale)')
plt.ylabel('variance of the average (sigma² = 1)')
plt.legend()
plt.show()

# Example: rho = 0.2 and B = 100
print('sigma2 = 1, rho = 0.2, B = 100:', round(0.2 + 0.8 / 100, 3))

## 6. Bagging (bootstrap aggregation) and out-of-bag observations

1. Draw $B$ **bootstrap samples** (size $n$, with replacement) from the training data.
2. Fit one (deep) tree on each bootstrap sample.
3. Aggregate: regression → **average** the $B$ predictions; classification → vote or average probabilities
   (classic: majority vote; scikit-learn: mean class probability).

**Out-of-bag (OOB) observations:** the probability that a given observation is *not* drawn in $n$ draws is
$(1-1/n)^n \to e^{-1} \approx 0.37$. Each tree never sees about 37 % of the data; these OOB points give an internal error
estimate without an extra validation split – not a replacement for a final test set.

In [ ]:
# Probability that an observation is not drawn into a bootstrap sample
for n in [10, 100, 1000, 10000]:
    print('n =', n, ' (1 - 1/n)^n =', round((1 - 1 / n) ** n, 4))
print('Limit e^-1 =', round(np.exp(-1), 4))

# Simulation: one bootstrap sample of n = 1000 observations
rng = np.random.default_rng(0)
n = 1000
sample = rng.integers(0, n, n)  # draw n times with replacement
share_not_drawn = 1 - len(np.unique(sample)) / n
print('Simulation: share not in the sample =', round(share_not_drawn, 3))

## 7. Random forest (regression): bagging + random features

1. **Bootstrap:** draw $B$ samples of size $n$ with replacement.
2. **Random features:** at every split consider only $m$ of the $p$ features.
3. **Grow deep trees:** little or no pruning – low bias, high variance.
4. **Aggregate:** average (regression); vote or mean class probability (classification).

**Why it works:** random feature subsets **de-correlate** the trees (smaller $\rho$) → lower variance of the average
(section 5).

- Classic heuristics for $m$ (Breiman; Hastie et al.): $p/3$ for regression, $\sqrt p$ for classification.
  scikit-learn defaults: classifier `max_features='sqrt'`, regressor all features (`1.0`, i.e. bagging).
- `n_estimators`: more trees stabilise the ensemble; the gains plateau while the cost grows. More trees do not cause
  overfitting.
- **Feature importance:** the impurity-based importance (MDI, `feature_importances_`) is computed on the training data
  and can be biased (e.g. towards features with many possible split points) → prefer the **permutation importance**
  (decrease of the score on test data when a feature is shuffled).

## 8. Gradient boosting (regression): learning from errors

$$\hat f_m(x) = \hat f_{m-1}(x) + \nu\, h_m(x)$$

In words: new model = old model + a small step $\nu$ (learning rate) towards a new small tree $h_m$.

**What does the next tree learn?** With squared loss: the current errors (residuals) $y - \hat f_{m-1}(x)$. With other
losses: the negative gradient of the loss (*pseudo-residuals*).

Example (squared loss): $y = 10$, current prediction 7 → residual 3; the next tree predicts ≈ 3; with $\nu = 0.1$ the
model moves to $7 + 0.1 \cdot 3 = 7.3$.

**Additional explanation.** Boosting builds the trees *sequentially* (each tree depends on the previous ones), random
forests build them *independently*. The main hyperparameters are the learning rate $\nu$, the number of trees $M$ and the
depth of the trees; a small $\nu$ needs a large $M$. Unlike random forests, boosting **can overfit** when $M$ becomes too
large.

In [ ]:
# One boosting step by hand: prediction 7, observed 10, learning rate 0.1
print('One boosting step: 7 + 0.1 * (10 - 7) =', 7 + 0.1 * (10 - 7))

# Figure showing the fit of gradient boosting after 1, 5, 20 and 200 trees
x_grid = np.linspace(0, 10, 400).reshape(-1, 1)
plt.scatter(x1, y1, color='grey')
for n_trees in [1, 5, 20, 200]:
    gb = GradientBoostingRegressor(
        n_estimators=n_trees, learning_rate=0.1, max_depth=2, random_state=0
    )
    gb.fit(X1, y1)
    plt.plot(
        x_grid, gb.predict(x_grid), label='after ' + str(n_trees) + ' trees'
    )
plt.xlabel('x')
plt.ylabel('y')
plt.title('Gradient boosting: the fit improves step by step')
plt.legend()
plt.show()

## 9. Regression models compared: typical tendencies

| | linear regression | single tree | random forest | gradient boosting |
|---|---|---|---|---|
| relationships | linear (+ chosen terms) | non-linear, interactions | non-linear, interactions | non-linear, interactions |
| interpretability | high (coefficients) | high (rules) | low (importance only) | low |
| complexity / overfitting | depends on number & choice of predictors | high variance if grown deep → prune | averaging substantially reduces variance | controlled by depth, learning rate $\nu$, iterations $M$ |
| data needed | small | medium | larger | larger |
| computation | very cheap | cheap | moderate, parallel | moderate, sequential |
| key hyperparameters | — (or regularisation) | `max_depth`, `ccp_alpha` | `n_estimators`, `max_features` | `learning_rate`, `n_estimators`, `max_depth` |

These are typical tendencies / rules of thumb – not fixed properties. **Compare candidate models with cross-validation on
your data.** Tree-based models cannot extrapolate beyond the range of the training data.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')